This notebook (will) import~~s~~ & perform basic analysis of the shendure data using our package.

In [4]:
import sys
import os
import statsmodels.discrete.count_model as smdc
import patsy
from tensorzinb.tensorzinb import TensorZINB
from formulaic import Formula
import pandas as pd
data_root="/gpfs/gibbs/pi/reilly/tabula_data"

In [27]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
from dask import delayed

import socket

In [20]:
sh_dat=pd.read_csv(f"{data_root}/shendure/shendure_counts_grouped.txt",sep="\t")

In [22]:
modelspecs= pd.DataFrame({
    "Dataset": ["Shendure (0)", "Shendure (0)"],
    "Hardware": ["Statsmodels (0)", "Tensorzinb a100 (1)"],
    "DNA": ["No dna (0)", "No dna (0)"],
    "Main Equ Type": ["simple addition (0)", "simple addition (0)"],
    "Z equ type": ["replicate (0)", "replicate (0)"],
    "Code": ["00000", "01000"],
    "Equation main": [
        "umis_mpra_bc ~ C(cre_id) + C(cell_type) -1",
        "umis_mpra_bc ~ C(cre_id) + C(cell_type) -1"
    ],
    "Equation Z": ["C(rep_id)", "C(rep_id)"],
    "Broken_by":[None,None],
    "Time": ["", ""],
    "Performance": ["", ""]
})
modelspecs

,Dataset,Hardware,DNA,Main Equ Type,Z equ type,Code,Equation main,Equation Z,Broken_by,Time,Performance
0,Shendure (0),Statsmodels (0),No dna (0),simple addition (0),replicate (0),00000,umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),None,,
1,Shendure (0),Tensorzinb a100 (1),No dna (0),simple addition (0),replicate (0),01000,umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),None,,


In [56]:
@delayed
def create_matricies(main_form,zin_form,data):
    y, X=Formula(main_form).get_model_matrix(data,output='pandas')
    Z=Formula(zin_form).get_model_matrix(data,output='pandas')
    return(X, y, Z)


#print(create_matricies(main_form=modelspecs.iloc[0]["Equation main"],zin_form=modelspecs.iloc[0]["Equation Z"],data=sh_dat))

def create_cluster():
    """
    Makes a simple slurm cluster with some preset parameters.
    """

    cluster=SLURMCluster(
        cores=1,#cores per slurm job
        memory="10G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra=["-p ycga", f"--job-name=simclust","--time=00:30:00"]
    )

    client = Client(cluster)

    print(f"Cluster started. Monitor on {cluster.dashboard_link}")

    cluster.scale(jobs=1)
    #cluster.adapt(minimum_jobs=1, maximum_jobs=10)

    return cluster,client

def kill_cluster(client):
    client.shutdown()

#we will start up the cluster, run, kill for each, then get job statistics from sacct
#each cluster will get an ID & put it in the name of the job & save that ID for later 
#sacct summary.. 
#collect with subprocess query

def statsmodels_fit(row):
    print(f"[+] Fitting {modelspecs.iloc[row]["Code"]}")
    print(f"[+] Creating cluster")
    cluster,client=create_cluster()

    print("[+] Scattering data")
    sh_dat_fut = client.scatter(sh_dat, broadcast=True)

    print("[+] Creating matricies")
    mats = create_matricies(
        main_form=modelspecs.iloc[row]["Equation main"],
        zin_form=modelspecs.iloc[row]["Equation Z"],
        data=sh_dat_fut
    )

    X, y, Z=mats.compute()

    if modelspecs.iloc[row]['Broken_by'] is None:
        print("[+] Model not parallalizable.")


    kill_cluster(client)


    
        

In [65]:
statsmodels_fit(row=0)

scattering data.
Model not parallalizable.
creating matricies
        umis_mpra_bc
0                  0
1                  0
2                  0
3                  0
4                  0
...              ...
778243             0
778244             0
778245             1
778246             0
778247             1

[778248 rows x 1 columns]
